# CUDA reproducibility check — mitigation OLL fragility

We found the out-of-lung-localization (OLL) metric differs by ~0.10 between a CUDA run (Colab) and a CPU run (local) on the **same** checkpoints — larger than the mitigation effect itself. This notebook quantifies that by running the eval on CUDA **twice** (run A and run B) so we can see (a) whether CUDA reproduces *itself*, and (b) how far CUDA sits from the deterministic CPU reference.

**Important:** this uses the SAME checkpoints Jonathan evaluated on CPU — they are already committed in the repo at `notebooks/checkpoints/control.pt` and `masked.pt`. Do NOT retrain, or the comparison is confounded (device vs. retraining).

**Before running:** `Runtime → Change runtime type → GPU`.

**Send back:** `cuda_repro.zip` (12 CSVs: runA + runB × {pretrained,control,masked} × {auroc,oll}).

### 1. Clone + install

In [4]:
!git clone -b feat/mitigation https://github.com/su-andrew/cs229-shortcut-detection.git
%cd cs229-shortcut-detection
!pip install -q torch torchvision torchxrayvision grad-cam scikit-image scikit-learn pandas numpy matplotlib pyyaml

Cloning into 'cs229-shortcut-detection'...
remote: Enumerating objects: 261, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 261 (delta 3), reused 8 (delta 3), pack-reused 238 (from 1)
Receiving objects: 100% (261/261), 51.42 MiB | 27.89 MiB/s, done.
Resolving deltas: 100% (147/147), done.
/content/cs229-shortcut-detection/cs229-shortcut-detection


### 2. Confirm GPU

In [5]:
import torch
assert torch.cuda.is_available(), 'No GPU — Runtime → Change runtime type → GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


### 3. Mount Drive + unzip data\nThe checkpoints (`control.pt`, `masked.pt`) are already in the repo at `notebooks/checkpoints/` (committed), so you only need the **data zip** from Drive. Edit `DATA_ZIP` to its path.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

# <<< EDIT THIS to your data zip's Drive path >>>
DATA_ZIP = '/content/drive/MyDrive/chexpert_data.zip'

!unzip -q "$DATA_ZIP" -d data/

# checkpoints are already in the cloned repo (notebooks/checkpoints/)
import os
need = ['data/chexpert/PNG_valid', 'data/chexpert/metadata.csv',
        'notebooks/checkpoints/control.pt', 'notebooks/checkpoints/masked.pt']
missing = [p for p in need if not os.path.exists(p)]
assert not missing, 'Missing: ' + str(missing) + ' — check DATA_ZIP path / repo clone.'
print('all inputs present:', need)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
all inputs present: ['data/chexpert/PNG_valid', 'data/chexpert/metadata.csv', 'notebooks/checkpoints/control.pt', 'notebooks/checkpoints/masked.pt']


### 4. Run A — CUDA eval, all 4 labels → `runA/`

In [7]:
!python -m src.mitigation_eval --tag pretrained --num-labels 4 --device cuda --output-dir runA
!python -m src.mitigation_eval --tag control --checkpoint notebooks/checkpoints/control.pt --num-labels 4 --device cuda --output-dir runA
!python -m src.mitigation_eval --tag masked  --checkpoint notebooks/checkpoints/masked.pt  --num-labels 4 --device cuda --output-dir runA

If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/chex-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/chex-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
[pretrained] AUROC:
           label    auroc  n_pos  n_neg
     Atelectasis      NaN     35      0
    Cardiomegaly 0.850877     19     24
   Consolidation 0.857778     15     45
           Edema 0.860248     46     21
Pleural Effusion 0.876610     63     53
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/pspnet_chestxray_best_model_4.pth -O /root/.torchxrayvision/models_data/pspnet_chestxray_best_model_4.pth`
[██████████████████████████████████████████████████]
[pretrained] OLL:
           label  oll_pos_mean  oll_pos_lo  oll_pos_hi  oll_neg_mean  oll_fp_mean  oll_fn_mean  n_pos  n_neg  n_empty_mask  n_boot  ranked
    Cardiomegaly     

### 5. Run B — identical CUDA eval → `runB/` (tests CUDA-vs-CUDA reproducibility)

In [8]:
!python -m src.mitigation_eval --tag pretrained --num-labels 4 --device cuda --output-dir runB
!python -m src.mitigation_eval --tag control --checkpoint notebooks/checkpoints/control.pt --num-labels 4 --device cuda --output-dir runB
!python -m src.mitigation_eval --tag masked  --checkpoint notebooks/checkpoints/masked.pt  --num-labels 4 --device cuda --output-dir runB

[pretrained] AUROC:
           label    auroc  n_pos  n_neg
     Atelectasis      NaN     35      0
    Cardiomegaly 0.850877     19     24
   Consolidation 0.857778     15     45
           Edema 0.860248     46     21
Pleural Effusion 0.876610     63     53
[pretrained] OLL:
           label  oll_pos_mean  oll_pos_lo  oll_pos_hi  oll_neg_mean  oll_fp_mean  oll_fn_mean  n_pos  n_neg  n_empty_mask  n_boot  ranked
    Cardiomegaly      0.492219    0.427245    0.562493      0.714516     0.603970     0.741917     17     19             0    2000    True
   Consolidation      0.655468    0.605015    0.704760      0.769219     0.756829          NaN     13     34             0    2000    True
           Edema      0.637360    0.593679    0.683336      0.772862     0.730326     0.934011     44     19             0    2000    True
Pleural Effusion      0.550967    0.496641    0.606188      0.675845     0.610346     0.692841     57     40             0    2000    True
wrote runB/mitigation_pretr

### 6. Quick look — does CUDA reproduce itself (runA vs runB)?

In [9]:
import pandas as pd
for tag in ['pretrained', 'control', 'masked']:
    a = pd.read_csv(f'runA/mitigation_{tag}_oll.csv').set_index('label')['oll_pos_mean']
    b = pd.read_csv(f'runB/mitigation_{tag}_oll.csv').set_index('label')['oll_pos_mean']
    print(f'\n=== {tag} OLL: runA vs runB ===')
    for l in a.index:
        print(f'  {l:18s} {a[l]:.4f}  {b[l]:.4f}  (diff {b[l]-a[l]:+.4f})')


=== pretrained OLL: runA vs runB ===
  Cardiomegaly       0.4922  0.4922  (diff +0.0000)
  Consolidation      0.6555  0.6555  (diff +0.0000)
  Edema              0.6374  0.6374  (diff +0.0000)
  Pleural Effusion   0.5510  0.5510  (diff +0.0000)

=== control OLL: runA vs runB ===
  Cardiomegaly       0.4189  0.4189  (diff +0.0000)
  Edema              0.5380  0.5380  (diff +0.0000)
  Pleural Effusion   0.5195  0.5195  (diff +0.0000)
  Consolidation      0.5926  0.5926  (diff +0.0000)

=== masked OLL: runA vs runB ===
  Edema              0.4911  0.4911  (diff +0.0000)
  Cardiomegaly       0.4005  0.4005  (diff +0.0000)
  Consolidation      0.5898  0.5898  (diff +0.0000)
  Pleural Effusion   0.4620  0.4620  (diff +0.0000)


### 7. Zip both runs + download → send to Jonathan

In [10]:
!zip -qr /content/cuda_repro.zip runA runB && echo done
from google.colab import files
files.download('/content/cuda_repro.zip')

done


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>